# 🏗️ Notebook 1: Reminder / Alert — Requirements & Architecture

**Goal of this lab:** build the intuition for designing a service that fires
*billions* of reminders at the right moment — the kind of thing that powers
Google Calendar, medication reminders, "your package is arriving" pushes,
and anniversary emails.

In this first notebook we:

1. Clarify what the system must do (and what it must *not* do).
2. Do the "back-of-envelope" math so our design isn't accidentally 1000×
   too small or too big.
3. Sketch the high-level architecture and justify every box.


## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Then in VS Code: pick the `.venv` kernel (top-right of the notebook).
If it doesn't show up, run `Cmd+Shift+P` → **Reload Window**.

All code here uses only the Python standard library + `pydantic`. No
servers, no databases — everything runs in-process so you can step
through the ideas.


## 🤔 What problem are we solving?

Imagine a tiny script:

```python
import time
time.sleep(seconds_until_3pm)
send_push("Take your pills 💊")
```

That works for *one* reminder on *one* machine. It falls apart the moment
you have:

- **Many users** — a single process can't hold millions of `sleep`s.
- **Crashes** — `time.sleep` is gone forever when the process dies.
- **Time zones** — "8 AM Monday" means different UTC instants for different
  users. Daylight Saving Time changes the offset twice a year.
- **Multiple channels** — push, email, SMS, each with its own failure
  modes and rate limits.

A Reminder / Alert system is essentially a **durable, distributed,
time-accurate scheduler**. The rest of this lab explores how to build one.


## 📋 Requirements

We write these down *before* drawing boxes — every later decision should
map back to one of these bullets.

### Functional
- Schedule a one-off reminder at a specific time.
- Schedule a recurring reminder (daily, weekly, monthly).
- Cancel or reschedule a pending reminder.
- Deliver via one of: push, email, SMS.
- Let the user (or an admin) see delivery status / history.

### Non-functional
- **Accuracy**: fire within ~1 second of the requested time.
- **Durability**: a server crash must not lose reminders.
- **At-least-once delivery**: the user will never *miss* a reminder; they
  may occasionally see a duplicate (which the receiver can dedupe).
- **Scale**: 1B active reminders, peaks of 100k firings/second at the
  top of the hour.
- **Time-zone correct**: "every weekday at 8 AM" should keep meaning
  8 AM local time, even across DST boundaries.


## 🧮 Back-of-envelope estimation

Let's turn those numbers into concrete capacity budgets. We'll do it in
code so you can tweak the inputs and see what changes.


In [1]:
# Inputs — change these to see how the design budget shifts.
active_reminders = 1_000_000_000        # 1 billion scheduled
peak_fires_per_second = 100_000         # top-of-the-hour spike
avg_row_bytes = 200                      # user_id, fire_at, payload, ...
avg_payload_bytes = 120                  # "Take your pills" + metadata

storage_gb = active_reminders * avg_row_bytes / (1024**3)
peak_bandwidth_mbps = peak_fires_per_second * avg_payload_bytes * 8 / 1e6

# A single worker can comfortably process ~1k deliveries/s.
workers_needed = peak_fires_per_second // 1_000

print(f"Storage footprint       : {storage_gb:,.0f} GB")
print(f"Peak outbound bandwidth : {peak_bandwidth_mbps:,.0f} Mbps")
print(f"Workers at peak         : ~{workers_needed}")


Storage footprint       : 186 GB
Peak outbound bandwidth : 96 Mbps
Workers at peak         : ~100


**What this tells us:**

- ~**200 GB** of hot data: fits in a *sharded* relational DB (e.g. Postgres
  with 8–16 shards by `user_id`). No need for anything exotic.
- ~**100 Mbps** egress at peak: trivial, but the *fan-out to push/SMS
  providers* is the real bottleneck — their APIs rate-limit us.
- ~**100 workers** at peak: we want them horizontally scalable and
  stateless, so we can add more during spikes.


## 🧨 Why the "just use `cron` / `sleep`" approach breaks

Let's demo the naive approach so the motivation is visceral. We'll
"schedule" 5 reminders by spawning threads that each `sleep()`.


In [2]:
import threading, time

def naive_schedule(delay_s: float, message: str):
    def fire():
        time.sleep(delay_s)
        print(f"  [{time.strftime('%H:%M:%S')}] {message}")
    threading.Thread(target=fire, daemon=True).start()

print(f"Scheduling at {time.strftime('%H:%M:%S')} ...")
for i, delay in enumerate([0.2, 0.5, 0.1, 0.3, 0.4]):
    naive_schedule(delay, f"reminder #{i}")

time.sleep(0.8)
print("Done.")


Scheduling at 04:58:30 ...


  [04:58:30] reminder #2
  [04:58:30] reminder #0
  [04:58:30] reminder #3


  [04:58:30] reminder #4
  [04:58:30] reminder #1


Done.


It "works" for 5 reminders on one machine. Now picture it with:

- **1B reminders** → 1B threads? The OS falls over around ~10k.
- **Process restart** → every in-flight `sleep` is lost. No durability.
- **Cancellation** → how do you find and stop a specific thread?
- **Multiple servers** → who owns which reminder? Duplicate fires?

Every pain point above maps to a component we'll add in the next
notebooks: a **durable store**, a **scheduler**, **workers**, and a
**delivery queue** per channel.


## 🏛️ High-level architecture

```
                 ┌──────────────────┐
   clients ───▶  │   REST API       │    (stateless, horizontally scaled)
                 └────────┬─────────┘
                          │ INSERT / UPDATE / DELETE
                          ▼
                 ┌──────────────────┐
                 │ Reminder Store   │    sharded SQL, indexed on
                 │ (source of truth)│    (status, fire_at)
                 └────────┬─────────┘
                          │ SELECT … FOR UPDATE SKIP LOCKED
                          ▼
                 ┌──────────────────┐
                 │ Scheduler /      │    many replicas, each claims
                 │ Dispatcher       │    a batch of "due" reminders
                 └────────┬─────────┘
                          │ enqueue(delivery_id, channel, payload)
                          ▼
                 ┌──────────────────┐
                 │ Delivery Queues  │    one per channel, so a slow
                 │ (push|email|sms) │    SMS provider can't back up push
                 └─┬──────┬──────┬──┘
                   ▼      ▼      ▼
                 Push   Email   SMS         external providers (APNs,
               workers workers workers      FCM, SES, Twilio, …)
```

### Why this shape?

- **One job per box.** API validates, Store persists, Scheduler picks,
  Workers deliver. If SMS is slow, the API still responds instantly.
- **Stateless services scale out.** Add capacity by adding replicas.
- **Stateful stores are chosen per access pattern.** SQL for durability +
  transactional cancel; an optional Redis ZSET in front of it for the
  hottest "next-few-seconds" window (we build this in notebook 3).
- **Per-channel queues** give independent back-pressure + retries.

> 💡 **Interview tip:** always justify a box by tying it to a requirement.
> "I add a per-channel queue because SMS providers rate-limit and I don't
> want that to block push." Never draw a box you can't defend.


## 🧭 What's next

- **Notebook 2** — design the data model (`Reminder`, `Delivery`) and REST
  API. We build a small in-memory service so you can `POST` a reminder and
  watch it fire.
- **Notebook 3** — *deep dive* on the scheduling algorithm itself. We go
  **bad → best**: polling loop → min-heap priority queue → hashed time
  wheel, and cover retries, idempotency, and DST.
